# 섹터 뉴스 시험 실행

GitHub Actions 에 올리기 전에 Colab 에서 먼저 돌려 보는 노트북입니다.
키 없이도 구글 뉴스 RSS 로 동작하니, 일단 3번 셀까지 실행해 결과부터 확인하세요.

In [ ]:
# 1) 저장소 가져오기 — 아래 주소를 본인 저장소로 바꾸세요
REPO = "https://github.com/사용자명/research-dashboard.git"

!rm -rf research-dashboard
!git clone -q $REPO
%cd research-dashboard
!pip install -q -r requirements.txt

In [ ]:
# 2) 키 입력 (선택) — 그냥 엔터를 치면 건너뜁니다
import os, getpass
for name in ["NAVER_CLIENT_ID", "NAVER_CLIENT_SECRET", "GEMINI_API_KEY"]:
    value = getpass.getpass(f"{name} (없으면 엔터): ")
    if value.strip():
        os.environ[name] = value.strip()

In [ ]:
# 3) 수집 실행
!cd scripts && python fetch_news.py

In [ ]:
# 4) 결과 확인 — 노이즈가 많으면 config/sectors.yaml 의 queries 와 exclude 를 고치세요
import json
data = json.load(open("docs/data/news.json", encoding="utf-8"))
print(f"{data['updated_at']} · 총 {data['total']}건\n")
for s in data["sectors"]:
    print(f"[{s['name']}]")
    for it in s["items"]:
        dup = f" (+{it['dup_count']})" if it["dup_count"] else ""
        print(f"  · {it['title']}{dup}")
        print(f"    {it['summary']}")
        print(f"    {it['source']} {it['url']}")
    print()

In [ ]:
# 5) 텔레그램 시험 발송 (선택)
import os, getpass
os.environ["TELEGRAM_BOT_TOKEN"] = getpass.getpass("봇 토큰: ")
os.environ["TELEGRAM_CHAT_ID"] = input("chat id: ")
!cd scripts && python push_telegram.py